# Reshaping By Pivoting and Grouping

 | Method | Description |
|---|---|
| `pd.pivot_table(values=None, index=None, columns=None, aggfunc='mean', fill_value=None, margins=False, margins_name='All', dropna=True, observed=False, sort=True)` | Create a pivot table. `index` and `columns` accept Series, column names, `pd.Grouper`, or lists. `aggfunc` can be a function, list, or dict mapping columns to functions. Missing values replaced with `fill_value`. `margins=True` adds subtotals/totals. `dropna=False` keeps empty column combinations. `observed=True` shows only observed categories for categorical groupers. |

In [1]:
from IPython.display import display
import numpy as np
import pandas as pd

df = pd.DataFrame({'age':[15, 17, 10],'growth':[.5, .7, 1.2],'Name':['Paul', 'George', 'Ringo'] })
display(df.columns)
display(df.index)
display(df.info())


df0 = pd.DataFrame({
    "firm":    ["A", "A", "A", "A", "B", "B", "B", "B", "C", "C", "C", "C"],
    "year":    [2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023],
    "region":  ["East", "East", "East", "East", "West", "West", "West", "West", "East", "East", "East", "East"],
    "sales":   [120, 135, 135, 160, 90, 110, 105, 140, 120, np.nan, 130, 130],
    "profit":  [15, 18, 18, 25, 10, 12, 11, 20, 15, 16, 19, 19]
}).rename(index={i: f'id-{i}' for i in range(12) })

df0

Index(['age', 'growth', 'Name'], dtype='str')

RangeIndex(start=0, stop=3, step=1)

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     3 non-null      int64  
 1   growth  3 non-null      float64
 2   Name    3 non-null      str    
dtypes: float64(1), int64(1), str(1)
memory usage: 219.0 bytes


None

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-4,B,2020,West,90.0,10
id-5,B,2021,West,110.0,12
id-6,B,2022,West,105.0,11
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16


What is the average age by country for each employment status?
- Put the country in the index
- Have a column for each employment status
- Put the average age in each cell

In [ ]:
jb2.pivot_table(index='country_live', columns='employment_status',values='age', aggfunc='mean')

In [ ]:
# .unstack() convert a multi index series to data frame
display(jb2.groupby(['country_live', 'employment_status']).age.mean().round(2))
display('------')
display(jb2.groupby(['country_live', 'employment_status']).age.mean().round(2).unstack().head(3))

## Using a Custom Aggregation Function
| Method | Description |
|---|---|
| `.groupby(by=None, axis=0, level=None, as_index=True, sort=True, group_keys=True, observed=False, dropna=True)` | Return a `GroupBy` object grouped by `by` (column name, function, Series, `pd.Grouper`, or list). Use `as_index=False` to keep grouping keys as columns. `observed=True` only shows observed categories for categorical groupers. `dropna=False` keeps groups with no values. |
| `.stack(level=-1, dropna=True)` | Push a column level into the index. `level` selects the column level to stack (`-1` innermost). |
| `.unstack(level=-1, dropna=True)` | Push an index level into the columns. `level` selects the index level to unstack (`-1` innermost). |

In [83]:
# what is the percent of Emacs user in each country?
def per_emacs(ser):
    return ser.str.contains('Emacs').sum() / len(ser) * 100

jb2.pivot_table(index='country_live', values='ide_main', aggfunc=per_emacs).head(3)

,ide_main
country_live,
Algeria,0.000000
Argentina,4.102564
Armenia,0.000000


In [84]:
jb2.groupby('country_live')[['ide_main']].agg(per_emacs).head(3)

,ide_main
country_live,
Algeria,0.000000
Argentina,4.102564
Armenia,0.000000


## multiple aggregates


In [85]:
jb2.pivot_table(index='country_live', values='age', aggfunc=('min', 'max')).head(3)

,max,min
country_live,,
Algeria,60.0,18.0
Argentina,60.0,18.0
Armenia,60.0,18.0


In [86]:
jb2.groupby(['country_live']).age.agg(['min','max']).head(3)

,min,max
country_live,,
Algeria,18.0,60.0
Argentina,18.0,60.0
Armenia,18.0,60.0


## Per Column Aggregations

In [87]:
col_list=['age', 'company_size','nps_main_ide', 'python_years', 'years_of_coding']
jb2.pivot_table(index='country_live', values=col_list,aggfunc=('min', 'max')).head(3)

age       company_size      nps_main_ide      python_years  \
               max   min          max  min          max  min          max   
country_live                                                                
Algeria       60.0  18.0       5000.0  1.0         10.0  0.0         11.0   
Argentina     60.0  18.0       5000.0  1.0         10.0  3.0         11.0   
Armenia       60.0  18.0       5000.0  1.0         10.0  5.0          6.0   

                  years_of_coding       
              min             max  min  
country_live                            
Algeria       0.5            11.0  0.5  
Argentina     0.5            11.0  0.5  
Armenia       0.5            11.0  0.5

In [88]:
# What are the minimum and maximum ages and the average company size for each country?

jb3=jb2.groupby('country_live').agg({'age': ['min', 'max'], 
                                 'company_size': 'mean'}).head(3)
display(jb3.columns) # columns have multi index
display(jb3[('age', 'min')] ) # access the multi index
display(jb3['age']['min'])    # access the multi index


MultiIndex([(         'age',  'min'),
            (         'age',  'max'),
            ('company_size', 'mean')],
           )

country_live
Algeria      18.0
Argentina    18.0
Armenia      18.0
Name: (age, min), dtype: float64

country_live
Algeria      18.0
Argentina    18.0
Armenia      18.0
Name: min, dtype: float64

In [89]:
# identical
jb2.pivot_table(index='country_live',aggfunc={'age': ['min', 'max'],'company_size': 'mean'}).head(3)

age       company_size
               max   min         mean
country_live                         
Algeria       60.0  18.0   679.535714
Argentina     60.0  18.0   931.713287
Armenia       60.0  18.0   570.629630

>> customized name cannot be done in `pivot_table()`. Use a keyword parameter to pass in a tuple of the column and aggregation function. The keyword parameter will be turned into a (flattened) column name.

In [90]:
jb2.groupby('country_live', observed=True).agg(age_min=('age', 'min'),age_max=('age', 'max'),team_size_mean=('company_size', 'mean')).head(3)

,age_min,age_max,team_size_mean
country_live,,,
Algeria,18.0,60.0,679.535714
Argentina,18.0,60.0,931.713287
Armenia,18.0,60.0,570.629630


In [91]:
import pandas as pd

df = pd.DataFrame({
    'country_live': pd.Categorical(
        ['US', 'US', 'CA'],
        categories=['US', 'CA', 'UK']   # UK exists as category, but no rows
    ),
    'age': [20, 30, 40]
})

# observed=False (include all category levels)
print(df.groupby('country_live', observed=False).agg(age_min=('age', 'min')))
# observed=True (only categories that appear in data)
print(df.groupby('country_live', observed=True).agg(age_min=('age', 'min')))




              age_min
country_live         
US               20.0
CA               40.0
UK                NaN
              age_min
country_live         
US                 20
CA                 40


## Grouping by Hierarchy

In [92]:
jb2.pivot_table(index=['country_live', 'ide_main'],values='age', aggfunc=['min', 'max'], observed=True).head(3)

min   max
                               age   age
country_live ide_main                   
Algeria      Atom             18.0  60.0
             Eclipse + Pydev  18.0  30.0
             IDLE             18.0  50.0

In [93]:
jb2.groupby(['country_live', 'ide_main'],observed=True)[['age']].agg('mean').reset_index().head(3)

,country_live,ide_main,age
0,Algeria,Atom,32.0
1,Algeria,Eclipse + Pydev,23.0
2,Algeria,IDLE,34.8


## group by function
Define customized groups to work on.


**Groupby Methods and Operations (g means already grouped df)**
| Method | Description |
|---|---|
| Column access | Access a column by attribute or index operation. |
| `g.agg(func=None, *args, engine=None, engine_kwargs=None, **kwargs)` | Apply an aggregate `func` to groups. `func` can be a string, function (accepts a column and returns a reduction), a list, or a dict mapping column name to string/function/list. |
| `g.aggregate` | Same as `g.agg`. |
| `g.all(skipna=True)` | Collapse each group to `True` if all values are truthy (per group). |
| `g.any(skipna=True)` | Collapse each group to `True` if any values are truthy. |
| `g.apply(func, *args, **kwargs)` | Apply a function to each group. Function accepts the group (DataFrame) and returns a scalar, Series, or DataFrame. Returns Series, DataFrame (each Series as a row), or DataFrame (with group index as inner index), respectively. |
| `g.count()` | Count of non-missing values for each group. |
| `g.ewm(com=None, span=None, halflife=None)` | Return an Exponentially Weighted grouper. Specify `com`, `span`, or `halflife`. Needs further aggregation. |
| `g.expanding(min_periods=1, center=False, axis=0, method='single')` | Return an expanding window object. Set `min_periods`, `center`, and `method` (`'single'` or `'table'`). Needs further aggregation. |
| `g.filter(func, dropna=True, *args, **kwargs)` | Return the original DataFrame with filtered groups removed. `func` is a predicate that accepts a group and returns `True` to keep it. If `dropna=False`, groups that evaluate to `False` are filled with `NaN`. |
| `g.first(numeric_only=False, min_count=-1)` | Return the first row of each group. If `min_count>0`, groups with fewer rows are filled with `NaN`. |
| `g.get_group(name, obj=None)` | Return a DataFrame corresponding to the named group. |
| `g.groups` | Property: dict mapping group name to list of index values. (See `.indices`.) |
| `g.head(n=5)` | Return the first `n` rows of each group (keeps original index). |
| `g.idxmax(axis=0, skipna=True)` | Return index label of the maximum value for each group. |
| `g.idxmin(axis=0, skipna=True)` | Return index label of the minimum value for each group. |
| `g.indices` | Property: dict mapping group name to numpy array of index positions. (See `.groups`.) |
| `g.last(numeric_only=False, min_count=-1)` | Return the last row of each group. `min_count>0` fills insufficient groups with `NaN`. |
| `g.max(numeric_only=False, min_count=-1)` | Return the maximum row of each group. `min_count>0` fills insufficient groups with `NaN`. |
| `g.mean(numeric_only=True)` | Return the mean of each group. |
| `g.min(numeric_only=False, min_count=-1)` | Return the minimum row of each group. `min_count>0` fills insufficient groups with `NaN`. |
| `g.ndim` | Property: number of dimensions of the result. |
| `g.ngroup(ascending=True)` | Return a Series indexed like the original index with group number labels. |
| `g.ngroups` | Property: number of groups. |
| `g.nth(n, dropna=None)` | Take the `n`th row from each group. |
| `g.nunique(dropna=True)` | Return counts of unique values per group. |
| `g.ohlc()` | Return a DataFrame with open/high/low/close values for each group. |
| `g.pipe(func, *args, **kwargs)` | Apply `func` to each group (functional chaining). |
| `g.prod(numeric_only=True, min_count=0)` | Return product of values for each group. |
| `g.quantile(q=.5, interpolation='linear')` | Return quantile(s) for each group; pass list `q` to get inner index for each quantile. |
| `g.rank(method='average', na_option='keep', ascending=True, pct=False, axis=0)` | Return numerical ranks within each group. `method` handles ties (`'average'`, `'min'`, `'max'`, `'first'`, `'dense'`). `na_option` controls NaN handling (`'keep'`, `'top'`, `'bottom'`). |
| `g.resample(rule, *args, **kwargs)` | Create a resample object with time-frequency `rule`. Needs further aggregation. |
| `g.rolling(window_size)` | Create a rolling grouper. Needs further aggregation. |
| `g.sample(n=None, frac=None, replace=False, weights=None, random_state=None)` | Return a sample from each group (keeps original index). |
| `g.sem(ddof=1)` | Return the standard error of the mean for each group. Specify degrees of freedom with `ddof`. |
| `g.shift(periods=1, freq=None, axis=0, fill_value=None)` | Return shifted values for each group (keeps original index). |
| `g.size()` | Return a Series with the size (row count) of each group. |
| `g.skew(axis=0, skipna=True, level=None, numeric_only=False)` | Return skewness for numeric columns per group (unbiased). |
| `g.std(ddof=1)` | Return standard deviation for each group (`ddof` controls degrees of freedom). |
| `g.sum(numeric_only=True, min_count=0)` | Return sum for each group. |
| `g.tail(n=5)` | Return the last `n` rows of each group (keeps original index). |
| `g.take(indices, axis=0)` | Return rows at the given positions (relative to each group). |
| `g.transform(func, *args, **kwargs)` | Return a DataFrame with the same shape as original index. `func` receives a group and should return an object with same dimensions as the group. |
| `g.var(ddof=1)` | Return variance for each group (`ddof` controls degrees of freedom). |

In [94]:
def even_grouper(idx):
    return 'odd' if idx % 2 else 'even'

jb2.pivot_table(index=even_grouper, aggfunc='size')

even    27231
odd     27231
dtype: int64

In [95]:
jb2.groupby(even_grouper).size()

even    27231
odd     27231
dtype: int64

# More aggregation

## filtering parts of groups

You often group and aggregate but want to get the result in terms of the
original index, not the aggregated index. The `.transform` method will allow you
to preserve the original index. If you want to filter based on aggregated data
but keep the original index (sans filtered rows), use the `.filter` method on
the groupby object.

| Method | Description |
|---|---|
| `g.filter(func, dropna=True, *args, **kwargs)` | Return the original DataFrame but with filtered groups removed. `func` is a predicate that accepts a group and returns `True` to keep that group's values. If `dropna=False`, groups that evaluate to `False` are filled with `NaN`. |
| `g.transform(func, *args, **kwargs)` | Return a DataFrame with the original index. The `func` is passed each group and should return an object with the same dimensions as the group (so the result can be aligned back to the original index). |

>> **note**: I have to remove NaN in the category variable to return consistent result. Don't know why?

In [96]:
# way 1
jb3=jb2[~jb2.country_live.isna()]
countries_to_remove = (jb3.country_live.value_counts().lt(330).pipe(lambda ser: ser[ser]).index)
jb3[~jb3.country_live.isna()].query('~country_live.isin(@countries_to_remove)').iloc[:3,:2]

,age,are_you_datascientist
1,21.0,Yes
2,30.0,No
4,21.0,NaN


In [97]:
# way 1=2
jb3[~jb3['country_live'].isin(countries_to_remove)].iloc[:3,:2]

,age,are_you_datascientist
1,21.0,Yes
2,30.0,No
4,21.0,NaN


In [98]:
# way 3
# in one single line
jb2.groupby('country_live',observed=False, dropna=True).filter(lambda g: g.country_live.size >=330).iloc[:3,:2]

,age,are_you_datascientist
1,21.0,Yes
2,30.0,No
4,21.0,NaN


## Aggregations while Keeping Rows

| String | Description |
|---|---|
| 'all' | Returns True for every value if every value is truthy. |
| 'any' | Returns True for every value if any value is truthy. |
| 'backfill' | Backfills values for the group. |
| 'bfill' | Backfills values for the group. |
| 'count' | Count of non-NA values for the group. |
| 'cumcount' | Number of each item in the group starting at 0 (S). |
| 'cummax' | Cumulative maximum for each group. |
| 'cummin' | Cumulative minimum for each group. |
| 'cumprod' | Cumulative product for each group. |
| 'cumsum' | Cumulative sum for each group. |
| 'diff' | Subtract the previous row from each row. The group needs to be numeric. |
| 'ffill' | Forward fill each group. |
| 'fillna' | Fill in missing values for each group. Must specify method ('ffill' or 'bfill') or value parameter. |
| 'first' | First row for each group. |
| 'idxmax' | Index of the maximum value for each group. |
| 'idxmin' | Index of the minimum value for each group. |
| 'last' | Last row for each group. |
| 'mad' | Mean absolute deviation for each group. |
| 'max' | Maximum value for each group. |
| 'mean' | Mean value for each group. |
| 'median' | Mean of each group. |
| 'min' | Minimum value for each group. |
| 'nth' | Nth value for each group. Must specify the n parameter. |
| 'nunique' | Number of unique values for each group. |
| 'pad' | Synonym for 'ffill'. |
| 'pct_change' | Percent change from current row and previous for each group. The group needs to be numeric. |
| 'prod' | Product of each group. |
| 'quantile' | Median of each group. Specify q (0-1) to change the quantile. The group needs to be numeric. |
| 'rank' | Rank of each group. |
| 'sem' | Unbiased standard error of each group. |
| 'shift' | Shift each group row down. Can specify periods (default 1) or freq with date index. |
| 'size' | Size of each group. Only works for a group with a single column (not a dataframe). |
| 'skew' | Skew of each group. |
| 'std' | Standard deviation of each group. |
| 'sum' | Sum of each group. (Will add strings!) |
| 'var' | Variance of each group. |

In [99]:
# create a new column that is the size of the group
jb2.assign(country_responses=(jb2.groupby('country_live', observed=True).age.transform('size'))).head(3)

,age,are_you_datascientist,company_size,country_live,employment_status,first_learn_about_main_ide,how_often_use_main_ide,ide_main,is_python_main,job_team,...,missing_features_main_ide,nps_main_ide,python_years,python3_version_most,several_projects,team_size,use_python_most,years_of_coding,python3_ver,country_responses
0,30.0,NaN,1.0,NaN,Partially employed by a company / organization,Conference / User Group,Weekly,PyCharm Community Edition,Yes,Work as an external consultant or trainer,...,"No, it has all the features I need",3.0,3.0,Python 3_7,"Yes, I work on many different projects",1,Unknown,1.0,3.7,NaN
1,21.0,Yes,5000.0,India,Fully employed by a company / organization,School / University,Daily,VS Code,Yes,Work in a team,...,"No, it has all the features I need",8.0,3.0,Python 3_6,"Yes, I work on one main and several side projects",2,Software prototyping,3.0,3.6,2800.0
2,30.0,No,5000.0,United States,Fully employed by a company / organization,Friend / Colleague,Daily,Vim,Yes,Work on your own project(s) independently,...,"No, it has all the features I need",10.0,3.0,Python 3_6,"Yes, I work on one main and several side projects",NaN,DevOps / System administration / Writing autom...,3.0,3.6,3975.0


# Melting, Transposing, and Stacking Data

| Method | Description |
|---|---|
| `.melt(id_vars=None, value_vars=None, var_name=None, value_name='value', col_level=None, ignore_index=True)` | Return an unpivoted DataFrame: stack each column in `value_vars` into rows, keeping the `id_vars` columns. |
| `g.transform(func, *args, **kwargs)` | Return a DataFrame with the original index. `func` is passed each group and should return an object with the same dimensions as the group. |
| `pd.options.display.max_columns` | Property to set the maximum number of columns shown by pandas display. |
| `pd.options.display.min_rows` | Property to set the minimum number of rows shown by pandas display (controls truncation). |
| `.stack(level=-1, dropna=True)` | Push a column level into the index. `level` selects the column level (`-1` is innermost). |
| `.unstack(level=-1, dropna=True)` | Push an index level into the columns. `level` selects the index level (`-1` is innermost). |
| `.swaplevel(i=-2, j=-1, axis=0)` | Swap levels `i` and `j` of a MultiIndex (0 is outermost, -1 innermost). Specify `axis` to operate on index or columns. |
| `.reset_index(level=None, drop=False, col_level=0, col_fill='')` | Return a DataFrame with a new index (or removed level). `level` selects which level(s) to reset; `drop=True` drops index values instead of moving to columns. `col_level`/`col_fill` control placement for multi-level columns. |
| `.pipe(func, *args, **kwargs)` | Apply `func` to the DataFrame and return the function’s result (useful for method chaining). |

## melt and unmelting

In [106]:
scores = pd.DataFrame({
 'name':['Adam', 'Bob', 'Dave', 'Fred'],
 'age': [15, 16, 16, 15],
 'test1': [95, 81, 89, None],
 'test2': [80, 82, 84, 88],
 'teacher': ['Ashby', 'Ashby', 'Jones', 'Jones']})

scores.melt(id_vars=['name', 'age'],value_vars=['test1', 'test2'])

,name,age,variable,value
0,Adam,15,test1,95.0
1,Bob,16,test1,81.0
2,Dave,16,test1,89.0
3,Fred,15,test1,NaN
4,Adam,15,test2,80.0
5,Bob,16,test2,82.0
6,Dave,16,test2,84.0
7,Fred,15,test2,88.0


In [107]:
import io
import pandas as pd
data0 = '''name,age,test1,test2,teacher
Adam,15,95.0,80,Ashby
Bob,16,81.0,82,Ashby
Dave,16,89.0,84,Jones
Fred,15,,88,Jones'''
scores = pd.read_csv(io.StringIO(data0))
print(scores)                         

   name  age  test1  test2 teacher
0  Adam   15   95.0     80   Ashby
1   Bob   16   81.0     82   Ashby
2  Dave   16   89.0     84   Jones
3  Fred   15    NaN     88   Jones


In [108]:
# wide to long form
scores.melt(id_vars=['name', 'age'],  value_vars=['test1', 'test2'])

,name,age,variable,value
0,Adam,15,test1,95.0
1,Bob,16,test1,81.0
2,Dave,16,test1,89.0
3,Fred,15,test1,NaN
4,Adam,15,test2,80.0
5,Bob,16,test2,82.0
6,Dave,16,test2,84.0
7,Fred,15,test2,88.0


In [109]:
# rename variable to var_name and value to val_name
melted=scores.melt(id_vars=['name', 'age','teacher'], 
            value_vars=['test1', 'test2'],
            var_name='test',
            value_name='score')
print(melted)

   name  age teacher   test  score
0  Adam   15   Ashby  test1   95.0
1   Bob   16   Ashby  test1   81.0
2  Dave   16   Jones  test1   89.0
3  Fred   15   Jones  test1    NaN
4  Adam   15   Ashby  test2   80.0
5   Bob   16   Ashby  test2   82.0
6  Dave   16   Jones  test2   84.0
7  Fred   15   Jones  test2   88.0


Using `pivot_table` or `groupby` to unmelt.

In [110]:
melted.pivot_table(index=['name','age', 'teacher'],
                   columns='test',
                   values='score').reset_index()

test,name,age,teacher,test1,test2
0,Adam,15,Ashby,95.0,80.0
1,Bob,16,Ashby,81.0,82.0
2,Dave,16,Jones,89.0,84.0
3,Fred,15,Jones,NaN,88.0


In [111]:
melted.groupby(['name', 'age', 'teacher', 'test']).score.mean().unstack().reset_index()
# unstack() takes the last index level and turns it into columns.

test,name,age,teacher,test1,test2
0,Adam,15,Ashby,95.0,80.0
1,Bob,16,Ashby,81.0,82.0
2,Dave,16,Jones,89.0,84.0
3,Fred,15,Jones,NaN,88.0


## pulling categorical variables as columns


In [112]:
print(melted.pivot(columns='teacher', values='score'))

teacher  Ashby  Jones
0         95.0    NaN
1         81.0    NaN
2          NaN   89.0
3          NaN    NaN
4         80.0    NaN
5         82.0    NaN
6          NaN   84.0
7          NaN   88.0


In [113]:
def pack_as_int(ser):
    return (ser[~ser.isna()].reset_index(drop=True).astype('int64'))

# .apply(...) by default applies the function column by column.
# manipulate the columna Ashby and ignore NaN
print(melted.pivot(columns='teacher', values='score').apply(pack_as_int))

teacher  Ashby  Jones
0           95   89.0
1           81   84.0
2           80   88.0
3           82    NaN


## stacking and unstacking
`.unstack` moves an index level into the columns. Usually, we
use this operation on multi-index data, moving one of the indices into the
columns (creating hierarchical columns). The `.stack` method does the reverse, moving a multi-level column into the index.

In [114]:
melted.groupby(['age', 'name']).size()

age  name
15   Adam    2
     Fred    2
16   Bob     2
     Dave    2
dtype: int64

In [115]:
melted.groupby(['age', 'name']).size().unstack() # groupby creates 2 index
# move the last index to column

name,Adam,Bob,Dave,Fred
age,,,,
15,2.0,NaN,NaN,2.0
16,NaN,2.0,2.0,NaN


In [116]:
melted.groupby(['age', 'name']).size().unstack(0) 
# move the index 0 to column

age,15,16
name,,
Adam,2.0,NaN
Bob,NaN,2.0
Dave,NaN,2.0
Fred,2.0,NaN


## flattening hierarchical indexes and columns

In [117]:
melted.groupby(['age', 'name']).size().reset_index()

,age,name,0
0,15,Adam,2
1,15,Fred,2
2,16,Bob,2
3,16,Dave,2


In [118]:
melted.groupby(['age', 'name'],as_index=False).size()

,age,name,size
0,15,Adam,2
1,15,Fred,2
2,16,Bob,2
3,16,Dave,2


In [119]:
print(melted.groupby(['age', 'name'], observed=True)\
      .mean(numeric_only=True).unstack())

     score                  
name  Adam   Bob  Dave  Fred
age                         
15    87.5   NaN   NaN  88.0
16     NaN  81.5  86.5   NaN


In [120]:
def flatten_cols(df):
    cols = ['_'.join(map(str, vals)) for vals in df.columns.to_flat_index()]
    df.columns = cols
    return df

print(melted.groupby(['age', 'name'], observed=True)\
      .mean(numeric_only=True).unstack()\
        .pipe(flatten_cols))

     score_Adam  score_Bob  score_Dave  score_Fred
age                                               
15         87.5        NaN         NaN        88.0
16          NaN       81.5        86.5         NaN
